# 06 — Final Model Export & Web Demo Preparation

This notebook prepares the artifacts needed by `app.py` and verifies that the frozen final model can make predictions from form-style input.

In [1]:
from pathlib import Path
import sys, json, os

def _find_root():
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "data" / "bonbanh_usedcar_dataset.csv").exists() and (p / "src").exists():
            return p
    raise FileNotFoundError("Project root not found. Keep notebooks inside the CarValue_AI_Complete_Project folder.")

PROJECT_ROOT = _find_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("PROJECT_ROOT:", PROJECT_ROOT)


PROJECT_ROOT: /mnt/data/CarValue_AI_Complete_Project


In [2]:
import json
import pandas as pd
from src.carvalue_utils import (
    ensure_modeling_artifacts, build_web_options, load_bundle, make_user_input
)

model_df, reference_year = ensure_modeling_artifacts(PROJECT_ROOT)
final_path = PROJECT_ROOT / "models" / "final_model.joblib"
assert final_path.exists(), "Run notebook 05 first to select/freeze the final model."
model = load_bundle(final_path)

options = build_web_options(model_df, reference_year)
with open(PROJECT_ROOT / "artifacts" / "web_options.json", "w", encoding="utf-8") as f:
    json.dump(options, f, ensure_ascii=False, indent=2)
print("Web options saved.")


Web options saved.


In [3]:
# Example form-style prediction using a real combination from the dataset
sample = model_df.dropna(subset=["brand","model","year","origin","body_type","transmission","fuel_type","exterior_color","drivetrain"]).iloc[0]
web_input = make_user_input(
    reference_year=reference_year,
    brand=str(sample["brand"]),
    model=str(sample["model"]),
    year=int(sample["year"]),
    mileage_km=0 if pd.isna(sample["mileage_km"]) else float(sample["mileage_km"]),
    origin=str(sample["origin"]),
    body_type=str(sample["body_type"]),
    transmission=str(sample["transmission"]),
    fuel_type=str(sample["fuel_type"]),
    engine_size_l=0.0 if pd.isna(sample["engine_size_l"]) else float(sample["engine_size_l"]),
    exterior_color=str(sample["exterior_color"]),
    seats=5 if pd.isna(sample["seats"]) else float(sample["seats"]),
    drivetrain=str(sample["drivetrain"]),
)
pred = float(model.predict(web_input)[0])
display(web_input)
print("Predicted asking price:", f"{pred:,.0f} VND")
print("Actual listing price for this example:", f"{float(sample['price_vnd']):,.0f} VND")


,brand,model,origin,body_type,transmission,fuel_type,exterior_color,drivetrain,car_age,mileage_km,mileage_zero_flag,engine_size_l,seats
0,Audi,A5 Sportback 2.0,Nhập khẩu,Sedan,Số tự động,Xăng,Trắng,AWD - 4 bánh toàn thời gian,9,NaN,1,2.0,5.0


Predicted asking price: 681,839,584 VND
Actual listing price for this example: 799,000,000 VND


In [4]:
print("To launch the demo after installing requirements:")
print(f"  cd {PROJECT_ROOT}")
print("  streamlit run app.py")
print("The web app predicts ONLINE ASKING PRICE, not the final negotiated transaction price.")


To launch the demo after installing requirements:
  cd /mnt/data/CarValue_AI_Complete_Project
  streamlit run app.py
The web app predicts ONLINE ASKING PRICE, not the final negotiated transaction price.
